### github link: 
[\[www.github.com/markusvalland-sudo/Deliverable_4\]](https://github.com/markusvalland-sudo/deliverable_4)


### Video links
##### task 1
https://nmbu.cloud.panopto.eu/Panopto/Pages/Viewer.aspx?id=ff90ec96-af8d-4ebc-9b5d-b50e010322bf 
##### task 2
https://nmbu.cloud.panopto.eu/Panopto/Pages/Viewer.aspx?id=ea95f1b5-2fd3-40ff-b5b6-b50f00dcd7e7



## Task 1

In [1]:
"""Temperature class supporting Celsius, Fahrenheit and Kelvin."""


class Temperature:

    VALID_UNITS = ("C", "F", "K")
    TOL = 1e-9

    def __init__(self, value: float, unit: str = "C"):
        self._unit = self._validate_unit(unit)
        self._value = float(value)
        if self.to_kelvin() < -self.TOL:
            raise ValueError("Temperature cannot be below absolute zero (0 K).")

    @classmethod
    def _validate_unit(cls, unit: str) -> str:
        """Return the unit in upper case, or raise ValueError if invalid."""
        if not isinstance(unit, str) or unit.upper() not in cls.VALID_UNITS:
            raise ValueError(f"Invalid unit: {unit!r}. Use 'C', 'F' or 'K'.")
        return unit.upper()

    #  Read-only attributes 
    @property
    def value(self) -> float:
        return self._value

    @property
    def unit(self) -> str:
        return self._unit

    #  Conversions 
    def to_celsius(self) -> float:
        if self._unit == "C":
            return self._value
        if self._unit == "K":
            return self._value - 273.15
        return (self._value - 32) * 5 / 9  # from Fahrenheit

    def to_fahrenheit(self) -> float:
        if self._unit == "F":
            return self._value
        if self._unit == "C":
            return self._value * 9 / 5 + 32
        return (self._value - 273.15) * 9 / 5 + 32  # from Kelvin

    def to_kelvin(self) -> float:
        if self._unit == "K":
            return self._value
        if self._unit == "C":
            return self._value + 273.15
        return (self._value - 32) * 5 / 9 + 273.15  # from Fahrenheit

    def convert_to(self, target_unit: str) -> "Temperature":
        target = self._validate_unit(target_unit)
        converters = {
            "C": self.to_celsius,
            "F": self.to_fahrenheit,
            "K": self.to_kelvin,
        }
        return Temperature(converters[target](), target)

    # String representations
    def __repr__(self) -> str:
        return f"Temperature({self._value!r}, {self._unit!r})"

    def __str__(self) -> str:
        return f"{self._value:.2f} °{self._unit}"

    # Comparisons (done in Kelvin, with tolerance)
    def __eq__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented
        return abs(self.to_kelvin() - other.to_kelvin()) < self.TOL

    def __ne__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented
        return not self == other

    def __lt__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented
        return self.to_kelvin() < other.to_kelvin() and not self == other

    def __le__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented
        return self.to_kelvin() < other.to_kelvin() or self == other

    def __gt__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented
        return self.to_kelvin() > other.to_kelvin() and not self == other

    def __ge__(self, other):
        if not isinstance(other, Temperature):
            return NotImplemented
        return self.to_kelvin() > other.to_kelvin() or self == other




### Task 1 - test

In [2]:
"""Tests for the Temperature class. Run with: python -m pytest -v test_temperature.py"""

import pytest

from temperature import Temperature


# 1. Initialization - 
def test_default_unit():
    assert Temperature(20).unit == "C"  # default unit is celcius


def test_case_insensitive_unit():
    assert Temperature(20, "f").unit == "F" # turns lower case -> upper case


def test_read_only_attributes(): #value and unit are locked -> changes make error
    t = Temperature(20, "C")
    with pytest.raises(AttributeError):
        t.value = 30
    with pytest.raises(AttributeError):
        t.unit = "K"


#  2. Validation 
def test_invalid_unit(): # test for invald unit - return Valuerror
    with pytest.raises(ValueError):
        Temperature(10, "X")


def test_below_absolute_zero(): # value error below absolute 0 
    with pytest.raises(ValueError):
        Temperature(-274, "C")
    with pytest.raises(ValueError):
        Temperature(-460, "F")
    with pytest.raises(ValueError):
        Temperature(-1, "K")


#  3. Conversions  # check the convertions
@pytest.mark.parametrize("c, f, k", [ #runs test 5 times
    (-273.15, -459.67, 0.0),   # absolute zero 
    (-40, -40, 233.15),        # Celsius-Fahrenheit crossover
    (0, 32, 273.15),           # freezing point of water
    (37, 98.6, 310.15),        # human body temperature
    (100, 212, 373.15),        # boiling point of water
])
def test_conversions(c, f, k): 
    t = Temperature(c, "C")
    assert t.to_fahrenheit() == pytest.approx(f) # approx allows minor rounding errors
    assert t.to_kelvin() == pytest.approx(k)
    assert Temperature(f, "F").to_celsius() == pytest.approx(c)
    assert Temperature(k, "K").to_celsius() == pytest.approx(c)


#  4. Round-trip conversion
def test_round_trip():
    original = Temperature(25, "C")
    back = original.convert_to("F").convert_to("K").convert_to("C") #converts C to F to K to C

    assert back == original


    #  5. Cross-unit comparisons 
    def test_cross_unit_equality():
        assert Temperature(0, "C") == Temperature(32, "F") #checks thtat 0 C is 32 F
    
    
    def test_cross_unit_ordering(): #checks with different units. 
        assert Temperature(0, "C") < Temperature(274, "K")
        assert Temperature(50, "F") <= Temperature(10, "C")
        assert Temperature(100, "C") > Temperature(200, "F")
        assert Temperature(373.15, "K") >= Temperature(212, "F")
    
    
    #  6. Formatting 
    def test_str():
        assert str(Temperature(25, "C")) == "25.00 °C" # turn 25, C into 25.00 
    
    
    def test_repr():
        assert repr(Temperature(25, "C")) == "Temperature(25.0, 'C')"


In [3]:
import sys
!{sys.executable} -m pytest -v test_temperature.py

============================= test session starts ==============================
platform darwin -- Python 3.13.5, pytest-8.3.4, pluggy-1.5.0 -- /opt/anaconda3/bin/python
cachedir: .pytest_cache
rootdir: /Users/wiktormyrhagen/Documents/INf201/Deliverable_4
plugins: anyio-4.7.0
collected 15 items                                                             

test_temperature.py::test_default_unit PASSED                            [  6%]
test_temperature.py::test_case_insensitive_unit PASSED                   [ 13%]
test_temperature.py::test_read_only_attributes PASSED                    [ 20%]
test_temperature.py::test_invalid_unit PASSED                            [ 26%]
test_temperature.py::test_below_absolute_zero PASSED                     [ 33%]
test_temperature.py::test_conversions[-273.15--459.67-0.0] PASSED        [ 40%]
test_temperature.py::test_conversions[-40--40-233.15] PASSED             [ 46%]
test_temperature.py::test_conversions[0-32-273.15] PASSED                [ 53%]
t

### Task 2 - Fractions

The `Fraction` class is stored in `fraction.py`. The fraction is always kept in lowest terms with a strictly positive denominator (a negative sign is moved to the numerator, and a zero denominator raises `ZeroDivisionError`).

Special methods implemented: `__init__`, all six comparisons (`__eq__`, `__ne__`, `__lt__`, `__le__`, `__gt__`, `__ge__`), and in addition `__repr__`, `__str__`, `__hash__`, `__add__`/`__radd__`, `__sub__`/`__rsub__`, `__mul__`/`__rmul__`, `__truediv__`/`__rtruediv__`, `__neg__`, `__abs__`, `__float__` and `__bool__`.

In [4]:
"""Fraction class with integer numerator and a strictly positive denominator."""

from math import gcd


class Fraction:

    def __init__(self, numerator: int, denominator: int = 1):
        # bool is a subclass of int, but True/False make no sense as fraction parts
        for part in (numerator, denominator):
            if not isinstance(part, int) or isinstance(part, bool):
                raise TypeError("Numerator and denominator must be integers.")
        if denominator == 0:
            raise ZeroDivisionError("Denominator cannot be zero.")

        # Keep the denominator strictly positive by moving the sign to the numerator
        if denominator < 0:
            numerator, denominator = -numerator, -denominator

        # Store the fraction in lowest terms, e.g. 2/4 -> 1/2
        common = gcd(numerator, denominator)
        self._numerator = numerator // common
        self._denominator = denominator // common

    #  Read-only attributes
    @property
    def numerator(self) -> int:
        return self._numerator

    @property
    def denominator(self) -> int:
        return self._denominator

    @staticmethod
    def _as_fraction(other):
        """Turn an int into a Fraction; return None for unsupported types."""
        if isinstance(other, Fraction):
            return other
        if isinstance(other, int) and not isinstance(other, bool):
            return Fraction(other)
        return None

    # String representations
    def __repr__(self) -> str:
        return f"Fraction({self._numerator}, {self._denominator})"

    def __str__(self) -> str:
        if self._denominator == 1:
            return str(self._numerator)
        return f"{self._numerator}/{self._denominator}"

    # Comparisons (a/b ? c/d  <=>  a*d ? c*b, since b, d > 0)
    def __eq__(self, other):
        other = self._as_fraction(other)
        if other is None:
            return NotImplemented
        return (self._numerator == other._numerator
                and self._denominator == other._denominator)

    def __ne__(self, other):
        result = self.__eq__(other)
        if result is NotImplemented:
            return NotImplemented
        return not result

    def __lt__(self, other):
        other = self._as_fraction(other)
        if other is None:
            return NotImplemented
        return self._numerator * other._denominator < other._numerator * self._denominator

    def __le__(self, other):
        other = self._as_fraction(other)
        if other is None:
            return NotImplemented
        return self._numerator * other._denominator <= other._numerator * self._denominator

    def __gt__(self, other):
        other = self._as_fraction(other)
        if other is None:
            return NotImplemented
        return self._numerator * other._denominator > other._numerator * self._denominator

    def __ge__(self, other):
        other = self._as_fraction(other)
        if other is None:
            return NotImplemented
        return self._numerator * other._denominator >= other._numerator * self._denominator

    # Hashing (equal fractions must have equal hashes, needed since __eq__ is defined)
    def __hash__(self) -> int:
        return hash((self._numerator, self._denominator))

    # Arithmetic
    def __add__(self, other):
        other = self._as_fraction(other)
        if other is None:
            return NotImplemented
        return Fraction(self._numerator * other._denominator + other._numerator * self._denominator,
                        self._denominator * other._denominator)

    def __radd__(self, other):  # int + Fraction
        return self.__add__(other)

    def __sub__(self, other):
        other = self._as_fraction(other)
        if other is None:
            return NotImplemented
        return Fraction(self._numerator * other._denominator - other._numerator * self._denominator,
                        self._denominator * other._denominator)

    def __rsub__(self, other):  # int - Fraction
        other = self._as_fraction(other)
        if other is None:
            return NotImplemented
        return other.__sub__(self)

    def __mul__(self, other):
        other = self._as_fraction(other)
        if other is None:
            return NotImplemented
        return Fraction(self._numerator * other._numerator,
                        self._denominator * other._denominator)

    def __rmul__(self, other):  # int * Fraction
        return self.__mul__(other)

    def __truediv__(self, other):
        other = self._as_fraction(other)
        if other is None:
            return NotImplemented
        if other._numerator == 0:
            raise ZeroDivisionError("Cannot divide by a zero fraction.")
        return Fraction(self._numerator * other._denominator,
                        self._denominator * other._numerator)

    def __rtruediv__(self, other):  # int / Fraction
        other = self._as_fraction(other)
        if other is None:
            return NotImplemented
        return other.__truediv__(self)

    # Unary operators and conversions
    def __neg__(self):
        return Fraction(-self._numerator, self._denominator)

    def __abs__(self):
        return Fraction(abs(self._numerator), self._denominator)

    def __float__(self) -> float:
        return self._numerator / self._denominator

    def __bool__(self) -> bool:
        return self._numerator != 0


### Task 2 - test

In [5]:
"""Tests for the Fraction class. Run with: python -m pytest -v test_fraction.py"""

import pytest

from fraction import Fraction


#  1. Initialization
def test_default_denominator():
    f = Fraction(3)
    assert (f.numerator, f.denominator) == (3, 1)


def test_reduced_to_lowest_terms():
    f = Fraction(6, 8)
    assert (f.numerator, f.denominator) == (3, 4)


def test_negative_denominator_moves_sign():
    f = Fraction(1, -2)
    assert (f.numerator, f.denominator) == (-1, 2)
    g = Fraction(-1, -2)
    assert (g.numerator, g.denominator) == (1, 2)


def test_zero_numerator():
    f = Fraction(0, 5)
    assert (f.numerator, f.denominator) == (0, 1)


def test_read_only_attributes():
    f = Fraction(1, 2)
    with pytest.raises(AttributeError):
        f.numerator = 3
    with pytest.raises(AttributeError):
        f.denominator = 3


#  2. Validation
def test_zero_denominator():
    with pytest.raises(ZeroDivisionError):
        Fraction(1, 0)


@pytest.mark.parametrize("num, den", [(1.5, 2), (1, 2.0), ("1", 2), (True, 2)])
def test_non_integer_parts(num, den):
    with pytest.raises(TypeError):
        Fraction(num, den)


#  3. Comparisons
def test_equal():
    assert Fraction(1, 2) == Fraction(2, 4)
    assert Fraction(4, 2) == 2  # comparison with int


def test_not_equal():
    assert Fraction(1, 2) != Fraction(1, 3)
    assert Fraction(1, 2) != 1


def test_ordering():
    assert Fraction(1, 3) < Fraction(1, 2)
    assert Fraction(1, 2) <= Fraction(2, 4)
    assert Fraction(3, 4) > Fraction(2, 3)
    assert Fraction(2, 3) >= Fraction(4, 6)
    assert Fraction(-1, 2) < 0
    assert Fraction(5, 2) > 2


def test_compare_with_other_type():
    # == with an unsupported type falls back to identity, giving False
    assert (Fraction(1, 2) == "1/2") is False
    assert Fraction(1, 2).__eq__("1/2") is NotImplemented
    with pytest.raises(TypeError):
        Fraction(1, 2) < "1/2"


#  4. Arithmetic
def test_add():
    assert Fraction(1, 2) + Fraction(1, 3) == Fraction(5, 6)
    assert Fraction(1, 2) + 1 == Fraction(3, 2)
    assert 1 + Fraction(1, 2) == Fraction(3, 2)


def test_sub():
    assert Fraction(1, 2) - Fraction(1, 3) == Fraction(1, 6)
    assert 1 - Fraction(1, 4) == Fraction(3, 4)


def test_mul():
    assert Fraction(2, 3) * Fraction(3, 4) == Fraction(1, 2)
    assert 3 * Fraction(1, 6) == Fraction(1, 2)


def test_truediv():
    assert Fraction(1, 2) / Fraction(1, 4) == 2
    assert 1 / Fraction(2, 3) == Fraction(3, 2)
    with pytest.raises(ZeroDivisionError):
        Fraction(1, 2) / Fraction(0, 1)


def test_unsupported_operand():
    with pytest.raises(TypeError):
        Fraction(1, 2) + 0.5


#  5. Unary operators and conversions
def test_neg_and_abs():
    assert -Fraction(1, 2) == Fraction(-1, 2)
    assert abs(Fraction(-3, 4)) == Fraction(3, 4)


def test_float():
    assert float(Fraction(1, 4)) == pytest.approx(0.25)


def test_bool():
    assert bool(Fraction(1, 2)) is True
    assert bool(Fraction(0, 3)) is False


def test_hash():
    assert hash(Fraction(1, 2)) == hash(Fraction(2, 4))
    assert len({Fraction(1, 2), Fraction(2, 4), Fraction(1, 3)}) == 2


#  6. Formatting
def test_str():
    assert str(Fraction(3, 4)) == "3/4"
    assert str(Fraction(4, 2)) == "2"


def test_repr():
    assert repr(Fraction(-2, 4)) == "Fraction(-1, 2)"


In [7]:
import sys
!{sys.executable} -m pytest -v test_fraction.py

============================= test session starts ==============================
platform darwin -- Python 3.13.5, pytest-8.3.4, pluggy-1.5.0 -- /opt/anaconda3/bin/python
cachedir: .pytest_cache
rootdir: /Users/wiktormyrhagen/Documents/INf201/Deliverable_4
plugins: anyio-4.7.0
collected 25 items                                                             

test_fraction.py::test_default_denominator PASSED                        [  4%]
test_fraction.py::test_reduced_to_lowest_terms PASSED                    [  8%]
test_fraction.py::test_negative_denominator_moves_sign PASSED            [ 12%]
test_fraction.py::test_zero_numerator PASSED                             [ 16%]
test_fraction.py::test_read_only_attributes PASSED                       [ 20%]
test_fraction.py::test_zero_denominator PASSED                           [ 24%]
test_fraction.py::test_non_integer_parts[1.5-2] PASSED                   [ 28%]
test_fraction.py::test_non_integer_parts[1-2.0] PASSED                   [ 32%]
t

### Use of AI
We used Claude (Anthropic) as a guide and assistant throughout this deliverable. We used it to:
Understand the assignment: We asked the AI to explain the requirements and suggest how to structure the work.
Generate and refine code: The AI provided suggestions and drafts for the classes and test files.
Debug: The AI helped us fix indentation errors, file naming problems, a Julia kernel selected by mistake in the notebook, problems with paths containing spaces, and setting up a virtual environment with pytest.
Understand the code: We asked for explanations of concepts such as @property, NotImplemented, pytest.raises, pytest.mark.parametrize and pytest.approx, so that we could explain every part of the code ourselves.
All code was run, tested and reviewed by us. We made sure we understood each part before including it, and we explain the code in our own words in the video. Our main takeaway is that AI is useful for getting started and for debugging, but we still had to understand the code ourselves to adapt it and make it work in our environment.
